# Step 3: SmoothQuant 量化（W8A8，两段式）+ 三方法对比 finale

**目标**：用 `llmcompressor` 的 **SmoothQuantModifier + GPTQModifier 两段式**把 Qwen2.5-7B-Instruct 量化成 **INT8 W8A8**（权重 + 激活都 8-bit）。SmoothQuant 通过数学等价的"平滑变换"把激活的离群点迁移到权重上，使**激活也能稳定地量化到 INT8**——这是它和 FP8（动态激活）/ AWQ（不量化激活）的根本区别。最后对比 FP8 / AWQ / SmoothQuant 三方法产物。

**对应 OUTLINE 课时**：2.5 SmoothQuant W8A8 全流程（~55 分钟）+ 2.6 三方法产物对比 finale。

> 为什么 SmoothQuant 要两段？第一段 `SmoothQuantModifier` 只是**改写模型权重**（插入平滑 scale，不量化）；第二段 `GPTQModifier(scheme="W8A8")` 才真把权重和激活压成 INT8。两段分开是因为"平滑"和"量化"是两个独立、可组合的步骤。

In [ ]:
%%capture
import subprocess, pathlib, json
import torch
import ipytest
ipytest.autoconfig()
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
from llmcompressor.modifiers.gptq import GPTQModifier

In [ ]:
# 解析仓库根目录（cwd 无关）：notebooks 通过 `uv run --directory envs/quant jupyter lab`
# 启动，但 jupyter 的 cwd 是所在 shell 的 cwd（不是 --directory 目标），所以
# 所有路径都从 git 仓库根派生，绝不依赖裸相对路径。
import subprocess, pathlib

REPO_ROOT = pathlib.Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
)
MODEL_DIR = REPO_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 download_model.sh 默认一致
TINY_MODEL_DIR = REPO_ROOT / "models" / "Qwen2.5-0.5B-Instruct"  # L3 先在 0.5B 上验，再上 7B
OUT_ROOT = REPO_ROOT / "out"                                  # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)
print("GPU OK" if __import__("torch").cuda.is_available() else "无 GPU（仅 L1/L2 可跑）")

## 原理：SmoothQuant 为什么能让激活"可量化"

INT8 量化激活的难点是**离群点**（少数极大的激活通道，详见 M1）。直接 per-token INT8 会被离群点撑爆 scale、压死多数值。SmoothQuant 的洞察：

- 激活难量化（有离群点），但权重好量化（分布平滑）。
- 用一个数学等价变换把"难"从激活搬到权重：对每个 channel，给激活乘 `s`、给对应权重列除 `s`，前向输出不变，但激活离群点被压小、权重被放大一点（权重本来就好量化，无所谓）。
- 平滑强度 `smoothing_strength`（记作 α）控制搬多少：α 越大搬越多。**llmcompressor 默认 α=0.8**（**不是**论文的 0.5，OUTLINE 2.5 明确提醒）。

公式（per-channel）：`s_j = max(|X_j|)^α / max(|W_j|)^(1-α)`，然后 `X'_j = X_j / s_j`，`W'_j = W_j * s_j`。

`scheme="W8A8"` = 权重 per-channel 对称 INT8 + 激活 **dynamic per-token** 对称 INT8。**必须先 SmoothQuant 平滑**否则 INT8 激活量化精度崩。

**易错点（OUTLINE 标注）**：
- `SmoothQuantModifier` 在 `llmcompressor.modifiers.transform.smoothquant`（**不是** `modifiers.smoothquant`）。
- `ignore=["lm_head"]` 必须是**列表**（写字符串会报错）。

## 本步填空

1. **`build_smoothquant_recipe(smoothing_strength, ignore)`** —— 构造两段式 recipe（SmoothQuantModifier + GPTQModifier），注意 `smoothing_strength` 默认 0.8。
2. **`smoothquant_config_summary(qc)`** —— 从 W8A8 产物抽出关键字段（INT8 权重 + INT8 动态激活，与 FP8 的 float 型对照）。
3. **`compare_methods(fp8_dir, awq_dir, sq_dir)`** —— finale：汇总三方法的位宽/显存/激活策略对比表（教学：理解三方法工程取舍）。

In [ ]:
def build_smoothquant_recipe(smoothing_strength=0.8, ignore=("lm_head",)):
    """返回 SmoothQuant 两段式 recipe（list of modifiers）。

    要求：
      - 第一段：SmoothQuantModifier(smoothing_strength=smoothing_strength)
      - 第二段：GPTQModifier(targets="Linear", scheme="W8A8", ignore=list(ignore))
    返回 [smooth_modifier, gptq_modifier]。

    注意：smoothing_strength 默认 0.8（llmcompressor 默认，非论文 α=0.5）；
          ignore 必须是 list（OUTLINE 提醒：写字符串会报错）。
    """
    # TODO: 构造并返回两个 modifier 组成的 list
    raise NotImplementedError


# 脚手架（提供）：真正跑 SmoothQuant 的 execution
def run_smoothquant_quantize(model, tokenizer, calib_texts, save_dir, n_samples=512, seq_len=2048):
    recipe = build_smoothquant_recipe()
    oneshot(
        model=model, tokenizer=tokenizer,
        dataset=_build_calib_dataset(calib_texts, n_samples),
        recipe=recipe, max_seq_length=seq_len, num_calibration_samples=n_samples,
    )
    save_dir = pathlib.Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir, save_compressed=True)  # save_compressed 写 INT8 packed
    return save_dir


# 脚手架（提供）：校准数据包装（SmoothQuant 需要校准，INT8 W8A8 至少 512 样本）
def _build_calib_dataset(calib_texts, n_samples):
    from datasets import Dataset
    if calib_texts is not None:
        return Dataset.from_dict({"text": list(calib_texts)[:n_samples]})
    from datasets import load_dataset
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    ds = ds.shuffle(seed=42).filter(lambda r: r["text"].strip())
    return ds.select(range(min(n_samples, len(ds))))

In [ ]:
def smoothquant_config_summary(quantization_config):
    """从 W8A8 (INT8) 产物的 quantization_config 抽出关键字段。

    返回 dict，至少含：
      - "quant_method"        : str
      - "weights_num_bits"    : int   （8）
      - "weights_type"        : str   （"int"，与 FP8 的 "float" 对照）
      - "weights_symmetric"   : bool
      - "input_dynamic"       : bool  （W8A8 激活 dynamic per-token）
      - "input_num_bits"      : int   （8）
      - "targets"             : list
    """
    # TODO: 解析并返回
    raise NotImplementedError

In [ ]:
def compare_methods(fp8_dir, awq_dir, sq_dir):
    """汇总 FP8 / AWQ / SmoothQuant 三方法产物对比。

    入参是三个产物目录路径（pathlib.Path 或 str，可能不存在——不存在则记 None）。
    返回 list of dict，每个 dict 一行：
      {"method": "FP8"/"AWQ"/"SmoothQuant",
       "weights_bits": int 或 None,
       "activations_bits": int 或 None（AWQ 不量化激活 -> None）,
       "disk_gb": float 或 None}
    依赖 smoothquant_config_summary（本步）与 s1/s2 的 summary 思路；这里直接读
    config_groups 的通用解析即可。
    """
    # TODO: 遍历三个目录，读 config.json -> quantization_config -> config_groups group_0，
    #       抽 weights.num_bits / input_activations.num_bits（无则 None），算 safetensors 总大小(GB)。
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_build_smoothquant_recipe_two_stages():
    recipe = build_smoothquant_recipe()
    assert isinstance(recipe, list) and len(recipe) == 2
    assert isinstance(recipe[0], SmoothQuantModifier)
    assert isinstance(recipe[1], GPTQModifier)

def test_build_smoothquant_recipe_default_strength_is_08():
    recipe = build_smoothquant_recipe()
    # llmcompressor 默认 smoothing_strength=0.8（非论文 0.5）
    assert recipe[0].smoothing_strength == 0.8

def test_build_smoothquant_recipe_w8a8_and_ignore_list():
    recipe = build_smoothquant_recipe(ignore=("lm_head",))
    assert recipe[1].scheme == "W8A8"
    assert recipe[1].ignore == ["lm_head"]
    assert isinstance(recipe[1].ignore, list)  # OUTLINE：ignore 必须是 list

def test_smoothquant_config_summary_on_fake_w8a8():
    fake = {
        "quant_method": "compressed-tensors",
        "config_groups": {"group_0": {
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "int", "symmetric": True},
            "input_activations": {"num_bits": 8, "dynamic": True, "type": "int"},
        }},
    }
    s = smoothquant_config_summary(fake)
    assert s["quant_method"] == "compressed-tensors"
    assert s["weights_num_bits"] == 8
    assert s["weights_type"] == "int"
    assert s["weights_symmetric"] is True
    assert s["input_dynamic"] is True
    assert s["input_num_bits"] == 8
    assert s["targets"] == ["Linear"]

def test_compare_methods_handles_missing_and_present(tmp_path):
    # 构造两个假产物目录 + 一个不存在的
    import json as _json
    def make_fake(d, w_bits, a_bits):
        d.mkdir(parents=True, exist_ok=True)
        cg = {"group_0": {"targets": ["Linear"],
              "weights": {"num_bits": w_bits, "type": "int" if w_bits==8 else "float"}}}
        if a_bits is not None:
            cg["group_0"]["input_activations"] = {"num_bits": a_bits, "dynamic": True}
        (d / "config.json").write_text(_json.dumps({"quantization_config": {"config_groups": cg}}))
        (d / "model.safetensors").write_bytes(b"x" * (2 * 1024 * 1024 * 1024))  # mock 2GB
    fp8 = tmp_path / "fp8"; make_fake(fp8, 8, 8)
    awq = tmp_path / "awq"; make_fake(awq, 4, None)
    rows = compare_methods(fp8, awq, tmp_path / "missing")
    by = {r["method"]: r for r in rows}
    assert by["FP8"]["weights_bits"] == 8 and by["FP8"]["activations_bits"] == 8
    assert by["AWQ"]["weights_bits"] == 4 and by["AWQ"]["activations_bits"] is None
    assert by["SmoothQuant"]["disk_gb"] is None  # 目录不存在

## L2：tiny 模型验证（CPU/GPU 秒级）

用 tiny Qwen2 真跑 SmoothQuant 两段式 + GPTQ W8A8。验证：① recipe 结构对 ② SmoothQuant 平滑后 INT8 激活量化真能跑 ③ 产物是 INT8（type="int"）。

In [ ]:
from transformers import Qwen2Config, Qwen2ForCausalLM, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

def make_tiny_tokenizer(vocab_size=320):
    vocab = {str(i): i for i in range(vocab_size)}
    tk = Tokenizer(WordLevel(vocab=vocab, unk_token="0"))
    tk.pre_tokenizer = Whitespace()
    return PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="0", pad_token="0",
                                   eos_token="0", bos_token="0", model_max_length=128)

def make_tiny_model(vocab_size=320, hidden_size=64):
    cfg = Qwen2Config(num_hidden_layers=2, hidden_size=hidden_size,
                      intermediate_size=hidden_size * 2, num_attention_heads=2,
                      num_key_value_heads=2, vocab_size=vocab_size, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()

tiny = make_tiny_model()
tiny_tok = make_tiny_tokenizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
tiny.to(device)
print("tiny SmoothQuant 模型就绪:", sum(p.numel() for p in tiny.parameters()), "params")

calib_texts = [" ".join(str(i % 50) for i in range(60))] * 8

tiny_out = OUT_ROOT / "tiny-smoothquant"
run_smoothquant_quantize(tiny, tiny_tok, calib_texts, tiny_out, n_samples=8, seq_len=64)

qc = json.loads((tiny_out / "config.json").read_text())["quantization_config"]
summary = smoothquant_config_summary(qc)
print("SmoothQuant 产物摘要:", summary)
assert summary["weights_num_bits"] == 8
assert summary["weights_type"] == "int"
assert summary["input_num_bits"] == 8
assert summary["input_dynamic"] is True
print("L2 PASS：tiny SmoothQuant 两段式跑通，产物为 INT8 W8A8")

## L3：H200 执行（真 Qwen2.5-0.5B 再 7B）

GPU 守卫：无 GPU 自动跳过。SmoothQuant W8A8 需要 512 校准样本（INT8 激活量化对样本量敏感）。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODEL_DIR)

    # 先 0.5B 快验
    m05b = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_05b = OUT_ROOT / "qwen05b-smoothquant"
    run_smoothquant_quantize(m05b, tok, None, out_05b, n_samples=128, seq_len=512)
    print("0.5B SmoothQuant done ->", out_05b)
    del m05b; torch.cuda.empty_cache()

    # 再 7B（INT8 W8A8 需 512 样本）
    m7b = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_7b = OUT_ROOT / "qwen7b-smoothquant"
    run_smoothquant_quantize(m7b, tok, None, out_7b, n_samples=512, seq_len=2048)
    print("7B SmoothQuant done ->", out_7b)
    del m7b; torch.cuda.empty_cache()
else:
    print("跳过：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查 + Finale：三方法对比

打印 SmoothQuant 7B 产物，并用 `compare_methods` 汇总 FP8 / AWQ / SmoothQuant 三方法的位宽、激活策略、显存。

In [ ]:
# SmoothQuant 自身产物
sq_summary = None
sq_dir = OUT_ROOT / "qwen7b-smoothquant"
if sq_dir.exists():
    qc = json.loads((sq_dir / "config.json").read_text())["quantization_config"]
    sq_summary = smoothquant_config_summary(qc)
    print("== SmoothQuant 7B ==", sq_summary)
    sq_size = sum(f.stat().st_size for f in sq_dir.glob("*.safetensors"))
    print("   safetensors: {:.2f} GB".format(sq_size / 1e9))
else:
    print("(SmoothQuant 7B 产物不存在，可能 L3 未跑)")

# Finale：三方法对比（依赖 s1/s2 产物存在）
print()
print("================ 三方法对比 finale ================")
rows = compare_methods(
    OUT_ROOT / "qwen7b-fp8",
    OUT_ROOT / "qwen7b-awq",
    OUT_ROOT / "qwen7b-smoothquant",
)
header = "{:<14}{:<10}{:<12}{:<14}{}".format("方法", "权重位宽", "激活位宽", "磁盘(GB)", "激活策略")
print(header)
for r in rows:
    disk = "{:.2f}".format(r["disk_gb"]) if r["disk_gb"] else "N/A"
    act = "dynamic" if r["activations_bits"] else "不量化(A16)"
    print("{:<14}{:<10}{:<12}{:<14}{}".format(
        r["method"], str(r["weights_bits"]), str(r["activations_bits"]), disk, act))

print()
print("要点：FP8 = float8 权重+激活；AWQ = int4 权重、激活不量化（最省显存）；")
print("      SmoothQuant = int8 权重+激活（激活靠平滑才可量化）。")
print("      三者产物都是 compressed-tensors，vLLM 直接加载。")